In [31]:
import psycopg2
import os 


# Database connection parameters 
db_name = "fl_election"
db_user = "postgres"
db_password = "1434"
db_host = "192.168.86.23" # Change to server's IP when acessing remotely
db_port = "5432"

def connect_to_db():
    """ Establish a connection to PostgreSQL and return the connection & cursor. """
    
    try:
        print("Attempting to connect...")
        conn = psycopg2.connect(
            dbname = db_name,
            user = db_user,
            password = db_password,
            host = db_host,
            port = db_port
        )
        cur = conn.cursor()
        print("Connected to PostgreSQL")
        return conn, cur
    except Exception as e:
        print(f"Database connection error: {e}")
        return None, None
    
# Call the function to test connection
conn, cur = connect_to_db()

Attempting to connect...
Connected to PostgreSQL


In [43]:
import pandas as pd

query = "select residence_address_1 street, residence_city_usps city, residence_zipcode zip_code, 'FL' AS state from voterfile.election_detail_2024"

df = pd.read_sql(query, conn)



/var/folders/h3/qc0834sx6gjb7174_439hbkc0000gn/T/ipykernel_38067/2907065733.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [57]:
import re 
import pandas as pd

def normalize_address(row):

    # lowercase + strip white space
    street = str(row['street']).strip().lower()
    street = re.sub(r'\s+', ' ', street)  # normalize extra spaces
    city = str(row['city']).strip().title()
    state = str(row['state']).strip().upper()
    zip_code = str(row['zip_code']).strip().upper().zfill(5)
    
    directionals = {
        'northwest' : 'nw',
        'northwest' : 'ne',
        'southwest' : 'sw',
        'southeast' : 'se',
        'north' : 'n',
        'south' : 's',
        'east' : 'e',
        'west' : 'w',
    }

    for full, abbr in directionals.items():
        street = re.sub(r'\b' + full + r'\b', abbr, street)

    # Normaization street suffixes 
    suffixes = {
        'street' : 'st',
        'avenue' : 'ave',
        'boulevard' : 'blvd', 
        'road' : 'rd', 
        'drive' : 'dr', 
        'lane' : 'ln', 
        'court' : 'ct'
     }
    for full, abbr in suffixes.items():
        street = re.sub(r'\b' + full + r'\b', abbr, street)
    
    # combined the final address string
    full_address = f"{street.title()}, {city}, {state} {zip_code}"
    return full_address
    

df['formatted_address'] = df.apply(normalize_address, axis=1)


In [67]:
df_test = df.head(200)  


In [69]:
import requests

def geocode_address(address, base_url = "http://192.168.86.36:8080"): 
    """ Geocode a single address using local Nominatim instance. """
    url = f"{base_url}/search"

    params = {
        "q" : address,
        "format" : "json", 
        "limit" : 1
    }

    try: 
        response = requests.get(url, params = params, timeout = 5)
        response.raise_for_status() # raise error if request failed
        date = response.json()
        if date: 
            return float(date[0]["lat"]), float(date[0]["lon"])
        else:
            return None, None
    except Exception as e: 
        print(f"Error for address: {address} - {e}")
        return None, None 

In [70]:
df_test[['lat', 'lon']] = df_test['formatted_address'].apply(
    lambda addr: pd.Series(geocode_address(addr))
)

/var/folders/h3/qc0834sx6gjb7174_439hbkc0000gn/T/ipykernel_38067/2950564567.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test[['lat', 'lon']] = df_test['formatted_address'].apply(


# Simple Testing API Calls 

In [60]:
import requests

address = "3717  Nw 55Th Pl, Gainesville, FL 32653"
url = f"http://192.168.86.36:8080/search"
params = {
    "q": address,
    "format": "json",
    "limit": 1
}

response = requests.get(url, params=params)
data = response.json()

if data:
    print("Latitude:", data[0]["lat"])
    print("Longitude:", data[0]["lon"])
else:
    print("No match found.")

Latitude: 29.705409
Longitude: -82.343083
